In [0]:
from pyspark.sql import SparkSession
import pyspark

In [0]:
spark = SparkSession.builder \
.appName("IcebergLocalSetup") \
.config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
.config("spark.sql.catalog.local.type", "hadoop") \
.config("spark.sql.catalog.local.warehouse", "file:///C:/iceberg_data_warehouse/w5/") \
.config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
.config("spark.jars", "file:///C:/iceberg/iceberg-spark-runtime-4.0_2.13-1.10.O.jar") \
.getOrCreate()

In [0]:
spark

In [0]:
print(pyspark.__version__)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS iceberg_schema

In [0]:
%sql
-- drop table lk_test_catalog.default.people_iceberg_tbl;
use catalog lk_test_catalog;
use schema iceberg_schema;
show tables;

In [0]:
current_catalog = spark.catalog.currentCatalog()
current_schema = spark.catalog.currentDatabase()
print(f"current catalog is {current_catalog} and schema is {current_schema}")

In [0]:
display(spark.sql("SELECT * FROM lk_test_catalog.iceberg_schema.people_iceberg_tbl.snapshots"))

In [0]:
spark.sql("""CREATE OR REPLACE TABLE  lk_test_catalog.iceberg_schema.people_iceberg_tbl
         ( id BIGINT,
          people_name STRING,
          age INT,
          city STRING
        )  USING ICEBERG
          """)

In [0]:
display(spark.sql("SHOW TBLPROPERTIES lk_test_catalog.iceberg_schema.people_iceberg_tbl"))

In [0]:
%sql
DESCRIBE EXTENDED lk_test_catalog.iceberg_schema.people_iceberg_tbl

In [0]:
spark.sql("""
ALTER TABLE lk_test_catalog.iceberg_schema.people_iceberg_tbl SET TBLPROPERTIES (
'write.metadata.delete-after-commit.enabled' = 'false',
'write.metadata-previous-versions-max'= 5,
'read.change.data.feed' = 'true' )
""")
spark.sql("SHOW TBLPROPERTIES lk_test_catalog.iceberg_schema.people_iceberg_tbl").show(truncate=False)

In [0]:
from pyspark.sql.functions import col, rand, concat, lit, expr

# List of Indian cities
cities = ["Mumbai", "Delhi", "Bangalore", "Hyderabad", "Ahmedabad", "Chennai", "Kolkata", "Pune", "Jaipur", "Lucknow"]

city_sql = ", ".join([f"'{city}'" for city in cities])
df = spark.range(10000, 110000) \
    .withColumn("people_name", concat(lit("Person_"), col("id"))) \
    .withColumn("age", (rand() * 60 + 18).cast("int")) \
    .withColumn("city", expr(f"element_at(array({city_sql}), cast(rand() * {len(cities)} + 1 as int))"))

df.writeTo("lk_test_catalog.iceberg_schema.people_iceberg_tbl").append()

In [0]:
ip_df = spark.sql("select count(1) from lk_test_catalog.iceberg_schema.people_iceberg_tbl")
display(ip_df)

In [0]:
%sql
DESCRIBE HISTORY lk_test_catalog.iceberg_schema.people_iceberg_tbl

In [0]:
ip_df = spark.sql("select * from lk_test_catalog.iceberg_schema.people_iceberg_tbl version as of 3")
display(ip_df)